In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# EXPORT: src/neural-network-from-scratch-python/lib/functions.py

class Logloss:
    def __init__(self):
        pass

    def forward(self, y_true, y_pred, eps=1e-15):
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.sum(y_true * np.log(y_pred))

    def gradient(self, y_true, y_pred, eps=1e-15):
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -y_true / np.sum(y_true * y_pred)


class Sigmoid:
    def __init__(self):
        pass

    def forward(self, x):
        return 1 / (1 + np.exp(-x))

    def gradient(self, x):
        sig = self.forward(x)
        return sig * (1 - sig)

class ReLU:
    def __init__(self):
        pass

    def forward(self, x):
        return np.maximum(0, x)

    def gradient(self, x):
        return (x > 0).astype(int)

class Softmax:
    def __init__(self):
        pass

    def forward(self, x):
        exp_x = np.exp(x - np.max(x))
        return exp_x / np.sum(exp_x)

    def gradient(self, x):
        s = self.forward(x).reshape(-1, 1)
        return np.diagflat(s) - np.dot(s, s.T)

In [ ]:
# EXPORT: src/neural-network-from-scratch-python/lib/neural_network.py

class NeuralNetwork:
    def __init__(
        self,
        layers_dimensions,
        input_dim,
        hidden_activation=ReLU(),
        loss=Logloss(),
        output_activation=None,
    ):
        self.training_cache = []
        self.error_cache = []
        self.val_error = []
        self.layers = len(layers_dimensions)
        self.layers_dimensions = layers_dimensions

        # First layer, hidden layers then output layer
        self.nn = [self.build_layer(layers_dimensions[0], input_dim)]
        self.nn.extend(
            [
                self.build_layer(
                    layers_dimensions[layer_dim], layers_dimensions[layer_dim - 1]
                )
                for layer_dim in range(1, self.layers)
            ]
        )
        self.bias = [np.zeros(layer_dim) for layer_dim in layers_dimensions]

        self.hidden_activation = hidden_activation
        self.output_activation = (
            output_activation if output_activation else hidden_activation
        )  # typically sigmoid or softmax but not mandatory
        self.loss = loss

    def build_layer(self, layer_dim, neuron_dim, limit=1):
        return np.array(
            [np.random.uniform(-limit, limit, neuron_dim) for _ in range(layer_dim)]
        )


    def get_values(
        self,
        training_cache=None,
        error_cache=None,
        epoch=None,
        batch=None,
        sample=None,
        layer=-1,
        after_activation=True,
        include_error=False,
        aggregate_error="mean",
        return_dict=False,
    ):
        """
        Unified accessor for cached forward pass values and errors.

        Parameters
        ----------
        training_cache : list
            Training cache (from self.training_cache or any compatible cache structure)
        error_cache : list
            Error cache (from self.error_cache)
        epoch, batch, sample, layer : int/list/None
            Indices to select. None = all, negative indices supported.
        after_activation : bool
            True -> post-activation values, False -> pre-activation values.
        include_error : bool
            If True, also return values from error_cache.
        aggregate_error : {"mean", "sum"}
            How to aggregate sample errors inside each batch (only returned, not plotted).
        return_dict : bool
            If True, always return a dict with keys {"values": ..., "errors": ...}.
            If False, return just values unless include_error=True.
        """

        # caches falls back to training ones when not provided
        if training_cache is None:
            training_cache=self.training_cache

        if include_error and error_cache is None:
            error_cache=self.error_cache

        act_idx = 1 if after_activation else 0

        # Helper to normalize indices
        def _indices(sel, length):
            if sel is None:
                return list(range(length))
            if isinstance(sel, (int, np.integer)):
                sel = [int(sel)]
            else:
                sel = [int(x) for x in sel]
            out = []
            for i in sel:
                if i < 0:
                    i = length + i
                if 0 <= i < length:
                    out.append(i)
                else:
                    raise IndexError(f"Index {i} out of range for length {length}")
            return out

        result = {}

        # -------------------------
        # 1) Collect training values
        # -------------------------
        values = []
        E = len(training_cache)
        for e in _indices(epoch, E):
            B = len(training_cache[e])
            for b in _indices(batch, B):
                S = len(training_cache[e][b])
                for s in _indices(sample, S):
                    L = len(training_cache[e][b][s])
                    for l in _indices(layer, L):
                        values.append(training_cache[e][b][s][l][act_idx])
        try:
            result["values"] = np.stack(values, axis=0)
        except Exception:
            result["values"] = np.array(values, dtype=object)

        # -------------------------
        # 2) Collect error values
        # -------------------------
        if include_error:
            errors = []
            E = len(error_cache)
            for e in _indices(epoch, E):
                B = len(error_cache[e])
                for b in _indices(batch, B):
                    S = len(error_cache[e][b])
                    for s in _indices(sample, S):
                        errors.append(error_cache[e][b][s])
            try:
                result["errors"] = np.stack(errors, axis=0)
            except Exception:
                result["errors"] = np.array(errors, dtype=object)

        # -------------------------
        # 3) Return
        # -------------------------
        if return_dict or include_error:
            return result

        return result.get("values", None)

    def plot_metrics(
        self,
        y_true,
        y_pred=None,
        training_cache=None,
        error_cache=None,
        epoch=None,
        batch=None,
        sample=None,
        display_confusion_matrix=False,
        aggregate_error="mean",
    ):
        """
        Plots both batch-wise error evolution and a confusion matrix.

        Parameters
        ----------
        y_true : array-like
            True labels (1D or one-hot encoded)
        y_pred : array-like, optional
            Predicted labels (1D or one-hot encoded). If None, will be computed from training_cache.
        training_cache : list, optional
            Neural network's training_cache (for computing predictions if y_pred is None)
        error_cache : list, optional
            Neural network's error_cache (for error chart)
        epoch, batch, sample : int/list/None
            Indices for selecting subset of cache to plot
        aggregate_error : {"mean", "sum"}
            How to aggregate sample errors for the error chart
        """

        # --------------------------
        # 1) Prepare y_true labels
        # --------------------------
        y_true = np.array(y_true)
        if y_true.ndim > 1 and y_true.shape[1] > 1:
            y_true_labels = np.argmax(y_true, axis=1)
        else:
            y_true_labels = y_true.flatten()

        # --------------------------
        # 2) Prepare y_pred labels
        # --------------------------
        
        # if y_pred is not provided, fallsback to predictions from training cache
        y_pred=np.argmax(self.get_values(self.training_cache, epoch=-1), axis=1) if y_pred is None else y_pred

        y_pred = np.array(y_pred)
        if y_pred.ndim > 1 and y_pred.shape[1] > 1:
            y_pred_labels = np.argmax(y_pred, axis=1)
        else:
            y_pred_labels = y_pred.flatten()

        # --------------------------
        # 3) Plot error evolution
        # --------------------------
        if error_cache is not None:
            errors=self.get_values(error_cache=self.error_cache, include_error=True)['errors']
            plt.figure(figsize=(10, 4))
            plt.plot(errors, marker="o")
            plt.title(f"Training error evolution ({aggregate_error} per batch)")
            plt.xlabel("Batch step")
            plt.ylabel("Error")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

            if self.val_error:
                plt.figure(figsize=(10, 4))
                plt.plot(self.val_error, marker="o")
                plt.title(f"Validation error evolution ({aggregate_error} per epoch)")
                plt.xlabel("Batch step")
                plt.ylabel("Error")
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

        # --------------------------
        # 4) Plot confusion matrix
        # --------------------------
        if display_confusion_matrix:
            cm=confusion_matrix(y_true, y_pred)
            n_classes = cm.shape[0]
        
            plt.imshow(cm, cmap="Blues", interpolation="nearest")
            plt.title("Confusion Matrix")
            plt.colorbar()
            
            classes = np.arange(n_classes)
            plt.xticks(classes, classes)
            plt.yticks(classes, classes)
            plt.xlabel("Predicted label")
            plt.ylabel("True label")
            
            # Annotate cells with counts
            for i in range(n_classes):
                for j in range(n_classes):
                    plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
            
            plt.tight_layout()
            plt.show()

    def forward(self, x, layer_n):  # forward from layer_n to layer_n + 1
        linear_transform = self.nn[layer_n].dot(x) + self.bias[layer_n]

        if layer_n == self.layers - 1:
            return linear_transform, self.output_activation.forward(linear_transform)

        return linear_transform, self.hidden_activation.forward(linear_transform)

    def backprop(self, batch_cache, X_batch, Y_batch, learning_rate):
        # dW : parameters error
        # dB : bias error
        dW, dB = (
            [np.zeros_like(W) for W in self.nn],
            [np.zeros_like(b) for b in self.bias],
        )

        for n, sample in enumerate(batch_cache):
            y_true = Y_batch[n]

            # last_z : last layer values (e.g. prediction) before activation
            # last_a : last layer values (e.g. prediction) after activation
            last_z, last_a = sample[-1]
            pred = last_a

            # d_loss/d_last_a (Loss gradient wrt prediction)
            da = self.loss.gradient(y_true, pred)

            # d_activation/d_last_z (Activation gradient wrt prediction)
            dz = da.dot(self.output_activation.gradient(last_z)).flatten()  # .flatten() as dz is a vector, not a matrix for multiclass classification

            # backprop
            for layer in range(self.layers - 1, -1, -1):
                # layer_input is the input of the current layer (e.g. the output of the previous one)
                # as X_batch does not contain the initial sample, we have to use an if statement
                if layer == 0:
                    layer_input = X_batch[n]
                else:
                    layer_input = sample[layer - 1][1]

                # np.outer(dz, layer_input) is
                dW[layer] += np.outer(dz, layer_input)
                dB[layer] += dz

                # propagate to previous layer
                if layer > 0:
                    da_prev = self.nn[layer].T @ dz
                    z_prev = sample[layer - 1][0]
                    dz = da_prev * self.hidden_activation.gradient(z_prev)

            # applying correction
            for layer in range(self.layers):
                self.nn[layer] -= learning_rate * dW[layer]
                self.bias[layer] -= learning_rate * dB[layer]
                
    def run(self, X, cache=False):
        x = X
        training_cache = [None] * self.layers if cache else None

        for layer in range(self.layers):
            out = self.forward(x, layer)
            if cache:
                training_cache[layer] = out
            x = out[1]  # post-activation value becomes input for next layer
        
        return x, training_cache

    def train(self, X_train, y_train, batch_size, epochs, learning_rate=0.01, X_val=None, y_val=None):
        if batch_size > X_train.shape[0]:
            raise Exception("batch_size > samples, exiting...")
            
        ohe=OneHotEncoder(sparse_output=False)
        
        if y_val is not None and X_val is not None:
            # one hot encoding for y_val (if provided)
            y_val=y_val.reshape(-1,1)
            y_val_ohe=ohe.fit_transform(y_val)

        # one hot encoding for y
        y_train=y_train.reshape(-1,1)
        y_train=ohe.fit_transform(y_train)

        # Stores cached forward pass values and errors for all epochs
        self.training_cache = []
        self.error_cache = []

        # Loop over epochs
        for e in range(epochs):
            current_line = 0  # Tracks position in dataset for batching
            epoch_training_cache = []  # Cache for this epoch
            epoch_error_cache = []     # Error for this epoch

            # Loop over dataset in batches
            while current_line < X_train.shape[0]:
                # Extract current batch
                batch_X = X_train[current_line : current_line + batch_size]
                batch_Y = y_train[current_line : current_line + batch_size]
                batch_training_cache = []  # Cache forward pass for this batch
                batch_error_cache = []     # Error for this batch

                # Process each sample individually
                for sample in range(batch_X.shape[0]):
                    x = batch_X[sample]

                    # Run forward pass and cache pre-activation & post-activation per layer
                    output, sample_training_cache = self.run(x, cache=True)
                    
                    # Save cached values and loss for this sample
                    batch_training_cache.append(sample_training_cache)
                    batch_error_cache.append(self.loss.forward(batch_Y[sample], output))

                # Perform backpropagation using cached forward pass and current batch
                self.backprop(batch_training_cache, batch_X, batch_Y, learning_rate)

                # Append batch caches to epoch-level caches
                epoch_training_cache.append(batch_training_cache)
                epoch_error_cache.append(batch_error_cache)

                # Move to the next batch
                current_line += batch_size

            if X_val is not None and y_val is not None:
                val_predictions=[]
                for sample in X_val:
                    val_predictions.append(self.run(sample)[0])
                
                self.val_error.append(self.loss.forward(y_val_ohe, np.array(val_predictions)))

            # Epoch completed, print progress
            print(f'Epoch {e + 1} / {epochs}')
            
            # Append epoch-level caches to global caches
            self.training_cache.append(epoch_training_cache)
            self.error_cache.append(epoch_error_cache)
    

    def test(self, X_test, y_test):
        # single forward pass to get predictions, no backprop
        predictions = []
        for x in X_test:
            y_pred, _ = self.run(x, cache=False)  # run without caching for test set
            predictions.append(y_pred)
        predictions = np.argmax(predictions, axis=1)

        # plot confusion matrix and error chart
        _=self.plot_metrics(y_test, predictions, display_confusion_matrix=True)
        print(classification_report(y_test, predictions))
        print(f'Error on test set: {self.loss.forward(y_test, predictions)}')
        
        return x  # return predictions for further evaluation

In [ ]:
# Usage example with load_digits dataset from sklearn

from sklearn.datasets import load_digits

digits = load_digits()

X = digits.data
y = digits.target

In [ ]:
n_inputs=X.shape[1]                # number of inputs, useful to define the size of the first layer
n_classes=np.unique(y).shape[0]    # number of classes, useful to define the size of the last layer

from sklearn.model_selection import train_test_split
X_tmp, X_test, y_tmp, y_test=train_test_split(X, y, test_size=.2, shuffle=True, random_state=42)
X_train, X_val, y_train, y_val=train_test_split(X_tmp, y_tmp, test_size=0.2)

nn = NeuralNetwork(
    layers_dimensions=[30, 20, 15, n_classes],  # can ajust network's shape, here: first layer has 30 neurons, second has 20, etc... last one has n_classes (don't touch the last one, it adapts automatically to the given dataset)
    input_dim=X.shape[1],                       # adapts automatically
    hidden_activation=Sigmoid(),                # can change for one of the defined activation functions in functions.py (or a custom one)
    loss=Logloss(),                             # can change for one of the defined loss functions in functions.py (or a custom one)
    output_activation=Softmax(),                # can change for one of the defined activation functions in functions.py (or a custom one)
)
nn.train(X_train, y_train, X_val=X_val, y_val=y_val, batch_size=5, epochs=5, learning_rate=0.01) # training step, ajust the batch_size, epochs amount and learning rate as needded

nn.plot_metrics(y_train, training_cache=nn.training_cache, error_cache=nn.error_cache, display_confusion_matrix=False)
preds=nn.test(X_test, y_test) # shows test results and store predictions in preds